In [1]:
# Step 0 : Impoting the tools 
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()
api_key=os.getenv("api_key")

client=Groq(api_key=api_key)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5052\1919957358.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 1 : Loading the documents(ingest)
pdf_loader=PyPDFDirectoryLoader(r"D:\Tesla-RAG-Project\data\tesla-annual-reports")
# Step 2 : Chunking 
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

tesla_chunks=pdf_loader.load_and_split(text_splitter)

In [3]:
# Step 3 : Store
embedding_model=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vectore_store=Chroma.from_documents(
    tesla_chunks,
    embedding_model,
    collection_name="tesla-10k-2019-to-2023",
    persist_directory="./tesla_db"

)

vectore_store.persist()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5052\795681704.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model=SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9214.12it/s]
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5052\795681704.py:12: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectore_store.persist()


In [4]:
vectorestore_persisted=Chroma(
    collection_name="tesla-10k-2019-to-2023",
    persist_directory="./tesla_db",
    embedding_function=embedding_model
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5052\3161850132.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorestore_persisted=Chroma(


In [5]:
# Step 4 : retreivement 
from pprint import pprint
query="What is the annual revenue in 2022"

retreiver=vectorestore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

docs=retreiver.invoke(query)
print(docs)

for i,r in enumerate(docs):
    print(f"Retreived Text : {i+1}")
    print(" ".join(r.page_content.split()))
    



[Document(metadata={'title': '', 'total_pages': 130, 'creationdate': '2024-01-29T11:11:14+00:00', 'producer': 'Qt 5.15.2', 'source': 'D:\\Tesla-RAG-Project\\data\\tesla-annual-reports\\tsla-20231231-gen.pdf', 'page': 33, 'creator': 'wkhtmltopdf 0.12.6', 'page_label': '34'}, page_content='Our\tcash\tflows\tprovided\tby\toperating\tactivities\tin\t2023\tand\t2022\twere\t$13.26\tbillion\tand\t$14.72\tbillion,\trespectively,\trepresenting\ta\tdecrease\tof\t$1.47\nbillion.\tCapital\texpenditures\tamounted\tto\t$8.90\tbillion\tin\t2023,\tcompared\tto\t$7.16\tbillion\tin\t2022,\trepresenting\tan\tincrease\tof\t$1.74\tbillion.\tSustained\ngrowth\thas\tallowed\tour\tbusiness\tto\tgenerally\tfund\titself,\tand\twe\twill\tcontinue\tinvesting\tin\ta\tnumber\tof\tcapital-intensive\tprojects\tand\tresearch\tand\ndevelopment\tin\tupcoming\tperiods.\n33'), Document(metadata={'title': '', 'producer': 'Qt 5.15.2', 'page_label': '42', 'creationdate': '2024-01-29T11:11:14+00:00', 'creator': 'wkhtmltopdf 0

In [6]:
# Step 5 : Generation 


# Choosing the model 
model_name='llama-3.3-70b-versatile'

# System Prompt
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer user questions only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.
If the answer is not found in the context, then don't make thing on your own
.
"""

# User Prompt
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""


# Query
user_input = "According to the text, what was the total revenue recognized by Tesla in 2023?"


relevent_chunks=retreiver.invoke(user_input)

context_list=[d.page_content for d in relevent_chunks]
context_for_query=" ".join(context_list)

prompt=[
    {"role":"system","content":qna_system_message},
    {"role":"user","content":qna_user_message_template.format(
        context=context_for_query,
        question=user_input
    )}
]

try :
    response=client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )
    prediction=response.choices[0].message.content.strip()

except Exception as e :
    prediction=f"Sorry , I encountered an error : {e}"

print(prediction)


$96,773 million


In [7]:

# Step 6 : Tesla-Inspired RAG Interface
import gradio as gr

def ask_tesla_rag(question):
    retrieved_chunks = retreiver.invoke(question)
    context = " ".join([doc.page_content for doc in retrieved_chunks])

    prompt = [
        {"role": "system", "content": qna_system_message},
        {"role": "user", "content": qna_user_message_template.format(
            context=context,
            question=question
        )}
    ]

    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    return response.choices[0].message.content

tesla_css = """
body {
    background: #0b0b0b;
}
.gradio-container {
    background: linear-gradient(180deg,#0b0b0b,#141414);
}
#title {
    text-align:center;
    color:white;
    font-size:42px;
    font-weight:bold;
    letter-spacing:4px;
}
"""

with gr.Blocks(css=tesla_css, theme=gr.themes.Base()) as demo:
    gr.Markdown("<h1 id='title'>TESLA RAG ENGINE</h1>")
    gr.Markdown(
        "Ask questions about Tesla annual reports (2019–2023)."
    )

    question = gr.Textbox(
        label="Question",
        placeholder="Example: What was Tesla's revenue in 2022?"
    )

    answer = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_btn = gr.Button("⚡ Ask Tesla AI")

    ask_btn.click(
        fn=ask_tesla_rag,
        inputs=question,
        outputs=answer
    )

demo.launch()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5052\652271665.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=tesla_css, theme=gr.themes.Base()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
